# Project 2: HMM Phoneme Recognizer — Duration-Augmented Model

## Overview

This notebook extends the baseline HMM phoneme recognizer with a **duration-augmented state space**. The key insight is that a plain phoneme-level HMM ignores how long the model has been in the current state, which limits its ability to model natural phoneme durations. By augmenting each state with a duration counter, the model can learn separate stay/switch probabilities for each (phoneme, duration) pair, producing more realistic phoneme boundary predictions.


## Notebook Roadmap
1. Configure environment and load data
2. Align time-stamped phoneme labels to frame-level labels
3. Build the duration-augmented state space
4. Estimate HMM parameters: π, A, μ, Σ
5. Implement augmented inference algorithms
6. Evaluate and compare against the baseline


## Section 1: Environment Setup and Data Loading

Load train and test split lists, match feature files to label files, and read all arrays into memory. The data paths are set for local execution — update `data_root` if running elsewhere.

In [ ]:
import numpy as np
from python_speech_features import mfcc, delta
import os
from pathlib import Path
import scipy.stats
from collections import defaultdict

In [ ]:
data_root = Path('/Users/emma/Documents/uni/project2/data')
feature_folder = data_root / 'out'
label_folder = data_root / 'cmu_us_slt_arctic' / 'lab'
txt_folder = data_root / 'txt'

In [ ]:
with open(txt_folder / 'train.txt', 'r', encoding='utf-8') as f:
    train_filenames = [line.rstrip('\n') for line in f]
with open(txt_folder / 'test.txt', 'r', encoding='utf-8') as f:
    test_filenames = [line.rstrip('\n') for line in f]
#with open(txt_folder / 'val.txt', 'r', encoding='utf-8') as f:
    #val_filenames = [line.rstrip('\n') for line in f]

In [ ]:
feature_files = [
    os.path.join(feature_folder, f)
    for f in os.listdir(feature_folder)
    if os.path.isfile(os.path.join(feature_folder, f)) and f.endswith('.npy')
]
feature_files = sorted(feature_files)

label_files = [
    os.path.join(label_folder, f)
    for f in os.listdir(label_folder)
    if os.path.isfile(os.path.join(label_folder, f)) and f.endswith('.lab')
]
label_files = sorted(label_files)

In [ ]:
train_feature_files = []
train_label_files = []
test_feature_files = []
test_label_files = []

train_filename_set = set(train_filenames)
test_filename_set = set(test_filenames)

for feature_idx, feature_filename in enumerate(feature_files):
    feature_basename = Path(feature_filename).stem

    if feature_basename in train_filename_set:
        train_feature_files.append(feature_filename)
        train_label_files.append(label_files[feature_idx])

    if feature_basename in test_filename_set:
        test_feature_files.append(feature_filename)
        test_label_files.append(label_files[feature_idx])

In [ ]:
def load_labels(label_file, arpabet_dic, n_frames, frame_step=0.01):
    label_array = np.genfromtxt(label_file, delimiter=" ", dtype=str)

    end_times = label_array[:, 0].astype(float)
    labels = label_array[:, 2]

    # frame times: 0.01, 0.02, ..., n_frames*frame_step
    frame_array = np.arange(1, n_frames + 1) * frame_step

    # for each end time, find how many frames are <= that boundary
    frame_ends = np.searchsorted(frame_array, end_times, side="left") + 1

    # convert cumulative frame counts into per-label counts
    counts = np.diff(np.r_[0, frame_ends])

    # repeat each label by its frame count
    states = np.repeat(labels, counts)

    # trim or pad to exactly n_frames
    states = states[:n_frames]

    if len(states) < n_frames:
        states = np.pad(states, (0, n_frames - len(states)), constant_values=labels[-1])

    states_num = np.array([arpabet_dic[p] for p in states], dtype=int)
    return states_num

## Section 2: Frame-Level Label Alignment

Convert each utterance from time-stamped phoneme boundaries (in seconds) into a frame-level label array, so that every MFCC feature vector has exactly one phoneme index.

`load_labels` uses `numpy.searchsorted` to map boundary times to frame indices, then repeats each label for its frame count. The result is trimmed or padded to match the feature array length exactly.

In [ ]:
ARPAbet_dic = {
    'aa': 0,
    'ae': 1,
    'ah': 2,
    'ao': 3,
    'aw': 4,
    'ay': 5,
    'b': 6,
    'ch': 7,
    'd': 8,
    'dh': 9,
    'eh': 10,
    'er': 11,
    'ey': 12,
    'f': 13,
    'g': 14,
    'hh': 15,
    'ih': 16,
    'iy': 17,
    'jh': 18,
    'k': 19,
    'l': 20,
    'm': 21,
    'n': 22,
    'ng': 23,
    'ow': 24,
    'oy': 25,
    'p': 26,
    'r': 27,
    's': 28,
    'sh': 29,
    't': 30,
    'th': 31,
    'uh': 32,
    'uw': 33,
    'v': 34,
    'w': 35,
    'y': 36,
    'z': 37,
    'zh': 38,
    'pau': 39,
    'ax': 40}
n_phonemes = len(ARPAbet_dic)

## Section 3: Duration-Augmented State Space

The baseline model treats each phoneme as a single state, which means it cannot distinguish between a phoneme that has just started and one that has been active for many frames. This section builds a richer state representation by augmenting each phoneme with a **duration counter** — the number of consecutive frames the model has already spent in that phoneme.

A state is now a pair **(phoneme, duration)**, capped at `max_duration` frames. Only states that actually appear in the training data are kept, giving a sparse state space that avoids allocating entries for impossible combinations.

In [ ]:
train_feature_list = []
for file in train_feature_files:
  feature_array = np.load(file)
  train_feature_list.append(feature_array)

test_feature_list = []
for file in test_feature_files:
  feature_array = np.load(file)
  test_feature_list.append(feature_array)

In [ ]:
train_label_list = []
for file_idx, label_file in enumerate(train_label_files):
  feature_array = train_feature_list[file_idx]
  n_frames = feature_array.shape[0]
  label_array = load_labels(label_file, ARPAbet_dic, n_frames)
  train_label_list.append(label_array)

test_label_list = []
for file_idx, label_file in enumerate(test_label_files):
  feature_array = test_feature_list[file_idx]
  n_frames = feature_array.shape[0]
  label_array = load_labels(label_file, ARPAbet_dic, n_frames)
  test_label_list.append(label_array)

In [ ]:
def state_duration(label_list, max_duration):
    augmented_label_list = []

    for utterance in label_list:
        n_frames = len(utterance)
        augmented = np.empty((n_frames, 2), dtype=int)

        duration = 0
        for t in range(n_frames):
            if t == 0:
                duration = 0
            elif utterance[t] == utterance[t - 1]:
                duration = min(max_duration, duration + 1) 
            else:
                duration = 0

            augmented[t, 0] = utterance[t]
            augmented[t, 1] = duration

        augmented_label_list.append(augmented)

    return augmented_label_list

In [ ]:
max_duration = 8

# Convert each training utterance from a plain phoneme sequence into
# a duration-augmented sequence of pairs: (phoneme_id, duration_in_state).
# Example: [aa, aa, aa, t, t] -> [(aa,0), (aa,1), (aa,2), (t,0), (t,1)]
state_alt = state_duration(train_label_list, max_duration)

# Collect every augmented state that actually appears in the training data.
# This gives a sparse state space: we only keep valid (phoneme, duration)
# pairs instead of assuming every phoneme has all durations 0..max_duration.
valid_states = sorted({
    (int(p), int(d))
    for utterance in state_alt
    for p, d in utterance
})

# Map each valid augmented state to a compact integer index.
# Example: state_to_idx[(phoneme_id, duration)] -> state_index
state_to_idx = {state: i for i, state in enumerate(valid_states)}

# Inverse mapping: from compact state index back to (phoneme, duration).
# idx_to_state[state_index] = [phoneme_id, duration]
idx_to_state = np.array(valid_states, dtype=int)

# Total number of valid augmented states in the sparse model.
n_states = len(valid_states)

# Dictionary mapping each phoneme to the list of augmented-state indices
# that belong to it.
# Example: phoneme_to_state_indices[p] = [indices for (p,0), (p,1), ...]
phoneme_to_state_indices = defaultdict(list)

# For each phoneme, store the largest duration that actually exists
# in the data. This is useful when defining "stay" transitions.
max_valid_duration = {}

for s, (p, d) in enumerate(valid_states):
    # Add this augmented state index to the list for phoneme p
    phoneme_to_state_indices[p].append(s)

    # Update the maximum observed duration for phoneme p
    max_valid_duration[p] = max(max_valid_duration.get(p, -1), d)

# Map each phoneme to its duration-0 augmented state index, if that state exists.
# This is useful for:
# - initializing the HMM
# - resetting duration when transitioning to a new phoneme
duration0_state = {}
for p in range(n_phonemes):
    if (p, 0) in state_to_idx:
        duration0_state[p] = state_to_idx[(p, 0)]

In [ ]:
# Dictionary that will count how many utterances start in each
# augmented state (phoneme, duration), including pauses.
start_state_counts = {}

# Loop over every duration-augmented utterance in the training set
for utterance in state_alt:
    # Ignore only completely empty utterances
    if len(utterance) == 0:
        continue

    # Take the very first augmented state, even if it is "pau"
    first_state = tuple(utterance[0])

    # Increment the count for this starting augmented state
    start_state_counts[first_state] = start_state_counts.get(first_state, 0) + 1

# Convert start-state counts into probabilities
total_starts = sum(start_state_counts.values())

# Initialize all augmented states with log-probability -inf
log_pi_aug = np.full(n_states, -np.inf)

# Convert start-state counts into probabilities
total_starts = sum(start_state_counts.values())

if total_starts > 0:
    for state, count in start_state_counts.items():
        if state in state_to_idx:
            s = state_to_idx[state]
            log_pi_aug[s] = np.log(count / total_starts)



### 3.1 Initial Distribution π

Estimated from the first frame of each training utterance, including utterances that start with silence (`pau`). Using the raw first frame gives an honest empirical estimate of what the model actually sees at t=0, rather than imposing an artificial prior.

The result is stored directly in log-space as `log_pi_aug`, one entry per augmented state.

In [ ]:
# create an array to keep the transition matrix
transition_matrix = np.zeros((n_phonemes, n_phonemes))
# loop over utterances 
for utterance in train_label_list:
    # loop over timeframes
    for t in range(len(utterance) - 1):
        # take state at time t
        current_state = utterance[t]
        # take state at time t+1
        next_state = utterance[t + 1]
        # keep count of phoneme pair occurence 
        transition_matrix[current_state, next_state] += 1

### 3.2 Structured Transition Matrix A

The transition matrix is built in two stages rather than estimated naively from raw counts.

**Stay vs. switch decomposition**

For each augmented state (phoneme, duration), we separately count:
- how often the next frame stays in the same phoneme (*stay*)
- how often it moves to a different phoneme (*switch*)

This gives a per-state stay/switch probability rather than a single shared value.

**Phoneme-to-phoneme switching distribution**

For switch transitions, we estimate which phoneme to switch *to* using a separate phoneme-level switching matrix, then combine it with the per-state switch probability.

**Structured stay transitions**

When staying, the duration counter advances by one (capped at the maximum observed duration for that phoneme). This means the model learns that longer stays become less likely as duration grows, which is linguistically realistic.

**Result:** a sparse (n_states × n_states) transition matrix where only structurally valid entries are non-zero.

In [ ]:
# Count how often each augmented state stays in the same phoneme
# versus switches to a different phoneme.
stay_counts = np.zeros(n_states)
switch_counts = np.zeros(n_states)

# Count phoneme-to-phoneme transitions, ignoring duration.
switching_matrix = np.zeros((n_phonemes, n_phonemes))

for utterance in state_alt:
    for t in range(len(utterance) - 1):
        # Current and next augmented states: (phoneme, duration)
        current_state = tuple(utterance[t])
        next_state = tuple(utterance[t + 1])

        current_phoneme, current_duration = current_state
        next_phoneme, next_duration = next_state

        # Compact index of the current augmented state
        state_idx = state_to_idx[current_state]

        if current_phoneme == next_phoneme:
            # Same phoneme -> staying transition
            stay_counts[state_idx] += 1
        else:
            # Different phoneme -> switching transition
            switch_counts[state_idx] += 1
            switching_matrix[current_phoneme, next_phoneme] += 1

In [ ]:
#Initialize the augmented-state transition matrix.
transition_matrix = np.zeros((n_states, n_states), dtype=float)

# For each augmented state, estimate:
# - probability of staying in the same phoneme
# - probability of switching to a different phoneme
stay_switch_probs = np.zeros((n_states, 2), dtype=float)
for s in range(n_states):
    total = stay_counts[s] + switch_counts[s]
    if total > 0:
        stay_switch_probs[s, 0] = stay_counts[s] / total
        stay_switch_probs[s, 1] = switch_counts[s] / total

# Normalize phoneme-to-phoneme switch counts row-wise.
switching_matrix_probs = np.divide(
    switching_matrix,
    switching_matrix.sum(axis=1, keepdims=True),
    out=np.zeros_like(switching_matrix, dtype=float),
    where=switching_matrix.sum(axis=1, keepdims=True) > 0
)

staying_probs = stay_switch_probs[:, 0]
switching_probs = stay_switch_probs[:, 1]

for s, (p, d) in enumerate(idx_to_state):
    # Stay transition: advance duration, or remain at the max valid duration
    next_d = min(d + 1, max_valid_duration[p])
    stay_state = state_to_idx[(p, next_d)]
    transition_matrix[s, stay_state] += staying_probs[s]

    # Switch transition: move to duration 0 of the next phoneme
    for next_p in range(n_phonemes):
        if next_p == p:
            continue
        if next_p not in duration0_state:
            continue

        next_s = duration0_state[next_p]
        transition_matrix[s, next_s] += (
            switching_probs[s] * switching_matrix_probs[p, next_p]
        )

In [ ]:
# Build a mask of allowed transitions
allowed_mask = np.zeros((n_states, n_states), dtype=bool)

for s, (p, d) in enumerate(idx_to_state):
    # Allowed stay transition
    next_d = min(d + 1, max_valid_duration[p])
    stay_state = state_to_idx[(p, next_d)]
    allowed_mask[s, stay_state] = True

    # Allowed switch transitions: to duration 0 of another phoneme
    for next_p in range(n_phonemes):
        if next_p == p:
            continue
        if next_p not in duration0_state:
            continue

        next_s = duration0_state[next_p]
        allowed_mask[s, next_s] = True
        
# Small additive smoothing value to avoid zero probabilities
# on transitions that are allowed by the model structure.
alpha = 0.001

# Start from the structured transition matrix estimated from counts.
transition_matrix_smooth = transition_matrix.copy()

# Add alpha only to transitions that are allowed.
# Impossible transitions remain exactly zero.
transition_matrix_smooth[allowed_mask] += alpha

# Compute row sums so each row can be turned back into a valid
# probability distribution.
normalisation_values = np.sum(transition_matrix_smooth, axis=1, keepdims=True)

# Normalize each row so outgoing transition probabilities sum to 1.
transition_matrix_norm = transition_matrix_smooth / normalisation_values

# Initialize all log-probabilities to -inf, which corresponds to log(0)
# for transitions that are impossible or have zero probability.
transition_matrix_log = np.full_like(transition_matrix_norm, -np.inf)

# Identify the entries with strictly positive probability.
positive_mask = transition_matrix_norm > 0

# Take logs only for positive probabilities to avoid log(0) warnings.
transition_matrix_log[positive_mask] = np.log(transition_matrix_norm[positive_mask])

### 3.3 Smoothing and Log Conversion

Laplace smoothing (α = 0.001) is applied **only to structurally allowed transitions** — impossible transitions remain exactly zero rather than receiving spurious probability mass. This is more principled than uniform smoothing over the full matrix.

After smoothing, each row is normalised to sum to 1, then converted to log-space for use in the inference algorithms.

In [ ]:
# define regularisation value alpha 
alpha = 0.001
# do regularisation 
transition_matrix_reg = transition_matrix + alpha
# get normalisation values 
normalisation_values = np.sum(transition_matrix_reg, axis=1, keepdims=True)
# normalise the probabilities 
transition_matrix_norm = transition_matrix_reg / normalisation_values
# take the log of all probabilities 
transition_matrix_log = np.where(transition_matrix_norm > 0, np.log(transition_matrix_norm), -np.inf)

### 3.4 Emission Parameters (μ, Σ)

One Gaussian emission model is estimated per **phoneme** (not per augmented state). All frames assigned to a phoneme across the training set are pooled to estimate the mean vector μ and covariance matrix Σ. A small diagonal regularisation term λI is added to Σ to ensure it is numerically invertible.

In [ ]:
# Create one list per phoneme.
# Each list will collect all feature vectors assigned to that phoneme
# across the entire training set.
feature_mapping = [[] for _ in range(n_phonemes)]

# Loop over all training utterances
for utterance_idx, utterance in enumerate(train_label_list):
    # Loop over all frame-level labels in the current utterance
    for label_idx, label in enumerate(utterance):
        # Extract the feature vector for this frame
        row = train_feature_list[utterance_idx][label_idx, :]

        # Append the feature vector to the list of the corresponding phoneme
        feature_mapping[label].append(row)

# Convert each phoneme's list of feature vectors into a NumPy array
# so it can be used easily for mean/covariance estimation later
feature_mapping = [np.array(rows) for rows in feature_mapping]

In [ ]:
# List that will store the Gaussian emission parameters for each phoneme:
# [mean_vector, covariance_matrix]
emission_parameters = []

# Small regularization constant added to covariance matrices
# to make them numerically stable and invertible
lam = 0.00001

# Dimensionality of each feature vector
feature_dim = train_feature_list[0].shape[1]

# Estimate one Gaussian distribution per phoneme
for phoneme in feature_mapping:
    if phoneme.shape[0] >= 2:
        # If we have at least 2 feature vectors, estimate both
        # the sample mean and sample covariance
        mean = np.mean(phoneme, axis=0)
        covariance = np.cov(phoneme, rowvar=False)

        # Add diagonal regularization for numerical stability
        covariance = covariance + np.eye(feature_dim) * lam

    elif phoneme.shape[0] == 1:
        # If we only have one example, estimate the mean from it,
        # but use a small diagonal covariance since sample covariance
        # cannot be estimated from a single point
        mean = np.mean(phoneme, axis=0)
        covariance = np.eye(feature_dim) * lam

    else:
        # If this phoneme has no training examples, fall back to
        # a zero mean and a small diagonal covariance
        mean = np.zeros(feature_dim)
        covariance = np.eye(feature_dim) * lam

    # Store the Gaussian parameters for this phoneme
    emission_parameters.append([mean, covariance])

## Section 4: Inference over the Augmented State Space

Both inference algorithms are adapted to operate over the augmented (phoneme, duration) state space. Emissions are still computed at the phoneme level and then broadcast to all augmented states belonging to that phoneme, since duration does not affect what the model observes — only how transitions are structured.

### 4.1 Forward Algorithm (Filtering)

Computes the posterior probability of each augmented state at every frame, using only past evidence.

$$\alpha_s(t) = P(O_{1:t},\, X_t = s)$$

The implementation works entirely in log-space, using `logsumexp` for the prediction step and normalising at each frame to prevent underflow. After decoding, augmented state indices are mapped back to phoneme indices via `idx_to_state[:, 0]`.

In [ ]:
def forward_filtering_augmented(
    observations,
    log_pi,          # shape (n_states,)
    log_A,           # shape (n_states, n_states)
    emission_params, # one Gaussian per phoneme
    idx_to_state     # shape (n_states, 2)
):
    import scipy.stats
    import scipy.special
    import numpy as np

    T = observations.shape[0]
    n_states = len(idx_to_state)
    n_phonemes = len(emission_params)

    log_alpha = np.full((T, n_states), -np.inf)

    # phoneme-level emissions
    log_emit_phoneme = np.zeros((T, n_phonemes))
    for p in range(n_phonemes):
        mu_p, sigma_p = emission_params[p]
        log_emit_phoneme[:, p] = scipy.stats.multivariate_normal.logpdf(
            observations, mean=mu_p, cov=sigma_p, allow_singular=True
        )

    # expand to sparse augmented states
    log_emit_all = np.zeros((T, n_states))
    for s, (p, d) in enumerate(idx_to_state):
        log_emit_all[:, s] = log_emit_phoneme[:, p]

    # init
    log_alpha[0, :] = log_pi + log_emit_all[0, :]
    log_alpha[0, :] -= scipy.special.logsumexp(log_alpha[0, :])

    # induction
    for t in range(1, T):
        for j in range(n_states):
            log_alpha[t, j] = log_emit_all[t, j] + scipy.special.logsumexp(
                log_alpha[t - 1, :] + log_A[:, j]
            )
        log_alpha[t, :] -= scipy.special.logsumexp(log_alpha[t, :])

    return log_alpha

#### 4.1.1 Single-Utterance Sanity Check

Run filtering on one test utterance to verify the output shape, that all predicted indices are valid phonemes, and to visually compare a slice of predictions against ground truth before full evaluation.

In [ ]:
# Pick the first test utterance
test_obs = test_feature_list[2]

alpha_result = forward_filtering_augmented(test_obs, log_pi_aug, transition_matrix_log, emission_parameters, idx_to_state)

#print(f"Output Shape: {alpha_result.shape}") # Should be (T, 41)
#print(f"Row sums (first 5): {np.sum(alpha_result[:5], axis=1)}") # Should all be 1.0

In [ ]:
# Get the index of the highest probability phoneme for each frame
pred_aug = np.argmax(alpha_result, axis=1)
predictions = idx_to_state[pred_aug, 0]

# Get the actual labels for the same utterance
ground_truth = test_label_list[2]

# Print a small slice to see if they look similar
print("Preds: ", predictions[50:70])
print("Truth: ", ground_truth[50:70])

In [ ]:
accuracy = np.mean(predictions == ground_truth)
print(f"Utterance 0 Frame Accuracy: {accuracy * 100:.2f}%")

### 4.2 Viterbi Decoding

Finds the single most likely augmented state sequence using dynamic programming, then maps back to phoneme indices.

$$\hat{X}_{1:T} = \arg\max_{X_{1:T}} P(X_{1:T} \mid O_{1:T})$$

All operations stay in log-space. The structured transition matrix means that most entries of `log_A` are `-inf`, so the max operation naturally respects the allowed transition topology without extra masking.

## Section 5: Evaluation

Frame-level accuracy, confusion matrices, and per-phoneme precision/recall are computed for the augmented filtering model on the train and test splits. Results can be compared directly against the baseline notebook to assess the impact of duration augmentation.

### 5.1 Evaluation Utilities

- `predict_filtering_labels` — runs augmented forward filtering and maps augmented state indices back to phoneme indices
- `evaluate_decoder` — runs a decoder over a split, prints throughput, returns aggregated predictions
- `compute_confusion_matrix` / `compute_precision_recall` — confusion statistics and per-phoneme metrics
- `plot_confusion_matrix` / `print_per_phoneme_table` — visualisation helpers
- `collapse_frame_labels` — run-length encodes a frame-level sequence into a phoneme sequence for qualitative inspection

In [ ]:
id_to_phoneme = {idx: phoneme for phoneme, idx in ARPAbet_dic.items()}
phoneme_names = [id_to_phoneme[i] for i in range(n_phonemes)]


def predict_filtering_labels(observations):
    posterior = forward_filtering_augmented(
        observations,
        log_pi_aug,
        transition_matrix_log,
        emission_parameters,
        idx_to_state 
    )
    return np.argmax(posterior, axis=1)   # augmented-state indices



def predict_viterbi_labels(observations):
    return viterbi(observations, initial_distribution_log, transition_matrix_log, emission_parameters)


def evaluate_decoder(feature_list, label_list, predict_fn, split_name, method_name, max_utterances=None,progress_every=1):
    from time import perf_counter

    n_total = len(feature_list)
    n_eval = n_total if max_utterances is None else min(max_utterances, n_total)

    y_true_all = []
    y_pred_all = []

    start_time = perf_counter()
    last_report_time = start_time
    last_report_idx = 0

    for idx in range(n_eval):
        observations = feature_list[idx]
        y_true = label_list[idx]
        y_pred = predict_fn(observations)

        if len(y_true) != len(y_pred):
            raise ValueError(
                f"Length mismatch in utterance {idx}: true={len(y_true)}, pred={len(y_pred)}"
            )

        y_true_all.append(y_true.astype(int))
        y_pred_all.append(y_pred.astype(int))

        if (idx + 1) % progress_every == 0 or (idx + 1) == n_eval:
            now = perf_counter()
            elapsed = now - start_time
            interval_elapsed = now - last_report_time
            interval_count = (idx + 1) - last_report_idx

            avg_utt_per_sec = (idx + 1) / elapsed if elapsed > 0 else 0.0
            recent_utt_per_sec = interval_count / interval_elapsed if interval_elapsed > 0 else 0.0

            print(
                f"{method_name} on {split_name}: {idx + 1}/{n_eval} "
                f"| avg {avg_utt_per_sec:.2f} utt/s "
                f"| recent {recent_utt_per_sec:.2f} utt/s "
                f"| elapsed {elapsed:.1f}s"
            )

            last_report_time = now
            last_report_idx = idx + 1

    y_true_flat = np.concatenate(y_true_all)
    y_pred_flat = idx_to_state[np.concatenate(y_pred_all), 0]

    total_elapsed = perf_counter() - start_time
    overall_utt_per_sec = n_eval / total_elapsed if total_elapsed > 0 else 0.0

    return {
        "split": split_name,
        "method": method_name,
        "n_utterances": n_eval,
        "n_frames": int(len(y_true_flat)),
        "frame_accuracy": float(np.mean(y_true_flat == y_pred_flat)),
        "elapsed_sec": float(total_elapsed),
        "utterances_per_sec": float(overall_utt_per_sec),
        "y_true": y_true_flat,
        "y_pred": y_pred_flat,
    }


def compute_confusion_matrix(y_true, y_pred, n_classes):
    confusion = np.zeros((n_classes, n_classes), dtype=np.int64)
    np.add.at(confusion, (y_true, y_pred), 1)
    return confusion


def compute_precision_recall(confusion):
    true_positive = np.diag(confusion).astype(float)
    predicted_positive = np.sum(confusion, axis=0).astype(float)
    actual_positive = np.sum(confusion, axis=1).astype(float)

    precision = np.divide(
        true_positive,
        predicted_positive,
        out=np.zeros_like(true_positive),
        where=predicted_positive > 0,
    )
    recall = np.divide(
        true_positive,
        actual_positive,
        out=np.zeros_like(true_positive),
        where=actual_positive > 0,
    )
    support = actual_positive.astype(int)
    return precision, recall, support


def print_per_phoneme_table(precision, recall, support, top_n=None):
    rows = []
    for i in range(n_phonemes):
        rows.append((phoneme_names[i], precision[i], recall[i], support[i]))

    rows.sort(key=lambda row: row[3], reverse=True)

    if top_n is None:
        top_n = len(rows)

    print(f"{'phoneme':>8}  {'precision':>9}  {'recall':>7}  {'support':>7}")
    for phoneme, prec, rec, sup in rows[:top_n]:
        print(f"{phoneme:>8}  {prec:9.3f}  {rec:7.3f}  {sup:7d}")


def plot_confusion_matrix(confusion, title):
    plt.figure(figsize=(10, 8))
    plt.imshow(confusion, cmap='Blues', aspect='auto')
    plt.colorbar(label='Frame count')
    ticks = np.arange(n_phonemes)
    plt.xticks(ticks=ticks, labels=phoneme_names, rotation=90, fontsize=7)
    plt.yticks(ticks=ticks, labels=phoneme_names, fontsize=7)
    plt.xlabel('Predicted phoneme')
    plt.ylabel('True phoneme')
    plt.title(title)
    plt.tight_layout()
    plt.show()


def collapse_frame_labels(frame_labels, phoneme_names, idx_to_state=None):
    if idx_to_state is None:
        symbols = [phoneme_names[int(x)] for x in frame_labels]
    else:
        symbols = [phoneme_names[int(idx_to_state[int(x), 0])] for x in frame_labels]

    collapsed = []
    for symbol in symbols:
        if not collapsed or symbol != collapsed[-1]:
            collapsed.append(symbol)

    return collapsed

In [ ]:
print(len(train_feature_list))

### 5.2 Frame Accuracy Evaluation

Set `max_eval_utterances = None` for full evaluation. The decoder prints throughput in utterances per second during the run and reports a frame accuracy summary table at the end.

In [ ]:
max_eval_utterances = 100
filter_train_eval = evaluate_decoder(
    train_feature_list,
    train_label_list,
    predict_filtering_labels,
    split_name='train',
    method_name='Filtering',
    max_utterances = max_eval_utterances
)

filter_test_eval = evaluate_decoder(
    test_feature_list,
    test_label_list,
    predict_filtering_labels,
    split_name='test',
    method_name='Filtering',
    max_utterances = max_eval_utterances
)

summary_rows = [
    filter_train_eval,
    filter_test_eval
]

print('Frame accuracy summary')
print(f"{'method':>10}  {'split':>7}  {'utt':>6}  {'frames':>8}  {'frame_acc':>10}  {'utt/s':>8}  {'time(s)':>8}")
for row in summary_rows:
    print(
        f"{row['method']:>10}  {row['split']:>7}  {row['n_utterances']:6d}  {row['n_frames']:8d}  "
        f"{row['frame_accuracy'] * 100:9.2f}%  {row['utterances_per_sec']:8.2f}  {row['elapsed_sec']:8.1f}"
    )

### 5.3 Confusion Matrix and Per-Phoneme Precision/Recall

The confusion matrix shows which phonemes are most frequently mistaken for each other. Per-phoneme precision and recall highlight whether errors are due to over-prediction (low precision) or under-detection (low recall). Requires Section 5.2 to have been run first.

In [ ]:
if 'filter_test_eval' not in globals():
    raise RuntimeError('Run the full evaluation cell in Section 5.2 before this cell.')

filter_test_confusion = compute_confusion_matrix(
    filter_test_eval['y_true'],
    filter_test_eval['y_pred'],
    n_phonemes,
)

print('Filtering confusion matrix shape:', filter_test_confusion.shape)

plot_confusion_matrix(filter_test_confusion, 'Filtering Confusion Matrix (Test)')

print('\nPer-phoneme precision/recall - Filtering')
f_precision, f_recall, f_support = compute_precision_recall(filter_test_confusion)
print_per_phoneme_table(f_precision, f_recall, f_support)

### 5.4 Qualitative Inspection: Predicted vs Ground Truth

Each test utterance is collapsed from frame-level to phoneme-level (consecutive identical frames merged) and displayed as a side-by-side comparison with the ground truth sequence. This reveals insertion, deletion, and substitution patterns that aggregate accuracy numbers obscure.